# TweetToxic – Detección de discurso de odio (HS · TR · AG) con modelos clásicos
**Curso de Procesamiento del Lenguaje Natural · Maestría en Inteligencia Artificial · Pontificia Universidad Javeriana**

Equipo: ____________________

Este notebook construye, de punta a punta, el modelo con el que se genera la submission `toxic_submission_v2.csv`. Para cada tweet en español se predicen tres etiquetas binarias:

| Etiqueta | Significado |
|---|---|
| **HS** (*Hate Speech*) | El texto contiene discurso de odio |
| **TR** (*Targeted*) | El odio va dirigido contra una persona concreta (y no contra un grupo genérico) |
| **AG** (*Aggressiveness*) | El texto es agresivo |

## Cumplimiento de las reglas
- Solo se usan librerías comunes de NLP/ML: `numpy`, `pandas`, `scipy`, `scikit-learn` y `nltk`. **No se usan transformers ni modelos preentrenados**: todo se aprende desde cero con `toxic_train.csv`.
- **El conjunto de prueba nunca participa en un `fit`**: los vectorizadores, escaladores y modelos se ajustan dentro de cada partición de la validación cruzada y, al final, solo con el train.
- La salida respeta el formato de `submission.csv` (`id,HS,TR,AG`).

## Resultados
**F1 medio** = F1 de la clase positiva de cada etiqueta, promediado sobre HS, TR y AG; estimado con validación cruzada de 5 particiones sobre el train (sección 8).

| Configuración | F1 HS | F1 TR | F1 AG | **F1 medio** | EMR |
|---|---|---|---|---|---|
| Regresión logística · TF-IDF de palabras (línea base) | 0,754 | 0,817 | 0,741 | 0,771 | 0,713 |
| + n-gramas de caracteres | 0,777 | 0,817 | 0,749 | 0,781 | 0,723 |
| + variables de léxico derivadas de las hipótesis | 0,793 | 0,823 | 0,791 | 0,803 | 0,755 |
| Ensamble LR + Powerset | 0,798 | 0,836 | 0,791 | 0,809 | 0,769 |
| **Stacking de 5 modelos (modelo final)** | **0,805** | **0,839** | **0,802** | **0,815** | 0,775 |

## Versiones enviadas
| Versión | Modelo | F1 medio (CV) | F1-macro 2 clases (CV) | Kaggle (público) |
|---|---|---|---|---|
| v1 · `toxic_submission.csv` | Ensamble LR + Powerset, elegido con el F1-macro de 2 clases | 0,809 | 0,847 | **0,78516** |
| v2 · `toxic_submission_v2.csv` | Stacking (metamodelo LR sobre CNB, LR, SVM, NB-LR y Powerset), elegido con el F1 medio | 0,815 | 0,850 | _pendiente_ |

La v1 se había evaluado con el F1-macro de dos clases (0,847), y por eso el 0,785 de Kaggle parecía bajo. Medido con el F1 de la clase positiva, el resultado esperado era 0,809, y el puntaje de Kaggle cae dentro de su rango de ruido para un leaderboard de ~577 filas (0,781-0,831). La sección 8.1 lo explica.

## Idea general
1. **Leer muchos tweets antes de modelar.** Se revisaron cerca de 900 tweets del train (todas las combinaciones de etiquetas minoritarias y muestras grandes de las mayoritarias). De esa lectura salen **11 hipótesis** que se verifican con datos en la sección 6.
2. **Preprocesar respetando los defectos del corpus**: las tildes y la ñ se perdieron en la exportación, y las palabras funcionales (`tú`, `eres`, `la`, `de`) cargan información, así que no se eliminan.
3. **Representar el texto en tres bloques**: TF-IDF de palabras, TF-IDF de n-gramas de caracteres y un bloque de variables de léxico construido directamente a partir de las hipótesis.
4. **Comparar modelos lineales** (regresión logística, SVM lineal, NB-LR, Naive Bayes) por etiqueta, además de un modelo *label powerset* que predice la combinación HS-TR-AG completa. Luego se prueban un promedio de modelos (ensamble) y un metamodelo que los combina (*stacking*), los umbrales se eligen con validación cruzada anidada y se aplica la restricción jerárquica TR ⊂ HS.

## 1. Librerías

Solo librerías permitidas por las reglas del curso. `nltk` se usa únicamente para la lista de *stopwords* del análisis descriptivo; el modelo **no** elimina stopwords (ver sección 5).

In [1]:
import os, re, sys, html, time, warnings
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, confusion_matrix
import nltk

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 150)
pd.set_option('display.width', 220)

## 2. Configuración

| Parámetro | Valor | Significado |
|---|---|---|
| `VALIDAR` | `True` | Corre toda la validación cruzada (ablación, modelos, ensamble y umbrales). Con `False` salta directo al modelo final con `ENSAMBLE_POR_DEFECTO` y umbral 0,5 |
| `N_FOLDS` | 5 | Particiones de la validación cruzada, estratificadas por la combinación HS-TR-AG |
| `C_LR`, `C_SVM` | 4,0 y 0,3 | Regularización de la regresión logística y del SVM lineal |
| `NB_ALPHA` | 1,0 | Suavizado del log-cociente de Naive Bayes en NB-LR |
| `PESO_LEX` | 0,3 | Peso del bloque de variables de léxico frente a los bloques TF-IDF |
| `UMBRAL_GRID` | 0,20 – 0,70 | Rejilla de umbrales de decisión por etiqueta |
| `CFG_WORD`, `CFG_CHAR` | ver celda | Palabras 1-2 (incluye tokens de una letra, p. ej. `t` de *tú*) y caracteres `char_wb` 2-5 |

La carpeta de datos se detecta automáticamente: `.claude/rules/data/` junto al notebook, `data/`, la carpeta del notebook o Kaggle.

In [2]:
# CONFIGURACIÓN
# =====================================================================================
VALIDAR      = True     # True: validación cruzada completa antes del modelo final
SEED         = 42
N_FOLDS      = 5
C_LR         = 4.0      # regularización de la regresión logística (binaria y powerset)
C_SVM        = 0.3      # regularización del SVM lineal
NB_ALPHA     = 1.0      # suavizado del log-cociente de NB-LR
PESO_LEX     = 0.3      # peso del bloque de léxico frente a los TF-IDF
UMBRAL_GRID  = np.round(np.arange(0.20, 0.701, 0.02), 2)
ENSAMBLE_POR_DEFECTO = ['LR', 'Powerset']   # se usa solo si VALIDAR = False
OUT_PATH     = 'toxic_submission_v2.csv'

CFG_WORD = dict(analyzer='word', ngram_range=(1, 2), min_df=2, sublinear_tf=True,
                token_pattern=r'(?u)\b\w+\b')
CFG_CHAR = dict(analyzer='char_wb', ngram_range=(2, 5), min_df=2, sublinear_tf=True)

LABELS = ['HS', 'TR', 'AG']
HERE = os.getcwd()
CANDIDATE_DIRS = [os.path.join(HERE, '.claude', 'rules', 'data'), os.path.join(HERE, 'data'),
                  os.path.join(HERE, 'tweet-toxic-2026-puj'), HERE, '/kaggle/input/tweet-toxic-2026-puj']
DATA_DIR = next((d for d in CANDIDATE_DIRS if os.path.exists(os.path.join(d, 'toxic_train.csv'))), None)
assert DATA_DIR is not None, 'No encontré toxic_train.csv: agregue su carpeta a CANDIDATE_DIRS.'

T0 = time.time()
def log(*m): print(f'[{time.time() - T0:6.0f}s]', *m, flush=True)
log('Datos en', DATA_DIR)

[     0s] Datos en C:\Users\jesus.romerom\Documents\Maestria\PLN\natural-language-processing-workshops\workshops\Kaggle-tweet-toxic\data


## 3. Lectura y calidad de los datos
Los archivos usan `;` como separador y están en **latin-1** (con UTF-8 la lectura falla). Antes de modelar se revisan tres defectos que aparecieron al leer los archivos en crudo.

In [3]:
train  = pd.read_csv(os.path.join(DATA_DIR, 'toxic_train.csv'), sep=';', encoding='latin-1')
test   = pd.read_csv(os.path.join(DATA_DIR, 'toxic_test.csv'),  sep=';', encoding='latin-1')
sample = pd.read_csv(os.path.join(DATA_DIR, 'submission.csv'))
train['text'] = train['text'].astype(str)
test['text']  = test['text'].astype(str)
# Una fila del train trae el id con una comilla sobrante ('"22621') por el mismo problema de escape del CSV
train['id'] = pd.to_numeric(train['id'].astype(str).str.strip('"'))
Y = train[LABELS].values
print('train', train.shape, '| test', test.shape, '| ejemplo de submission', sample.shape)
train.head()

train (1731, 5) | test (1154, 2) | ejemplo de submission (1154, 4)


,id,text,HS,TR,AG
0,20430,@SomosRedSaime amigos SAIME mi nombre es walter perez V-13.899.989 wperez001@gmail.com y desde el 16/03/2017 que fui a cita aun no me imprimen el...,0,0,0
1,22434,Preferan enfrentar los peligros de regresar a su hogar antes que ser encarcelados en estas condiciones seal la organizacin que tambin demand a la ...,0,0,0
2,23930,@srtaalegre_ Siiii jajjajajajjajajaja yo te iba a ensear pero vivs lejos nde puta.,0,0,0
3,20412,@anaisbernal Esta es la justicia en espaa para estos indeseables una puta verguenza,0,0,0
4,23531,Escucho Arabe y me dan ms ganas de estar en cheboli muviendo el orto con mis amigas we,0,0,0


In [4]:
RE_ERROR_EXCEL = re.compile(r'^#\W*NOM(BRE|E)\?$')

def calidad(df, nombre):
    t = df['text']
    return pd.Series({
        'filas': len(df),
        'textos con caracteres no ASCII': t.map(lambda s: any(ord(c) > 127 for c in s)).sum(),
        'error de Excel (#¿NOMBRE?)': t.str.strip().str.match(RE_ERROR_EXCEL).sum(),
        'filas entrecomilladas ("")': t.str.contains('"', regex=False).sum(),
        'textos duplicados': t.duplicated().sum(),
    }, name=nombre)

pd.concat([calidad(train, 'train'), calidad(test, 'test')], axis=1)

,train,test
filas,1731,1154
textos con caracteres no ASCII,0,6
error de Excel (#¿NOMBRE?),1,6
"filas entrecomilladas ("""")",39,25
textos duplicados,0,5


**Defecto 1: se perdieron las tildes y la ñ.** Ningún texto del train tiene caracteres no ASCII: las letras acentuadas se *borraron*, no se reemplazaron por su versión sin tilde. Por eso *cállate* aparece como `cllate`, *España* como `Espaa` y *organización* como `organizacin`, mientras que quien escribió sin tilde conserva la forma `callate`. La misma palabra queda con dos grafías.

In [5]:
grafias = {'cllate (cállate, tilde borrada)': r'\bcllate\b', 'callate (escrito sin tilde)': r'\bcallate\b',
           'Espaa (España)': r'\bespaa\b', 'organizacin (organización)': r'\borganizacin\b',
           'rabe (árabe)': r'\brabes?\b', 'gero (güero)': r'\bger[oa]s?\b'}
low = train['text'].str.lower()
pd.Series({k: int(low.str.contains(p).sum()) for k, p in grafias.items()}, name='tweets en train').to_frame()

,tweets en train
"cllate (cállate, tilde borrada)",158
callate (escrito sin tilde),211
Espaa (España),35
organizacin (organización),1
rabe (árabe),121
gero (güero),69


**Defecto 2: errores de Excel.** Algunas celdas se convirtieron en `#¿NOMBRE?` (o `#NOME?`), probablemente porque el texto original empezaba con `=` o `-`. Ese contenido no se puede recuperar: en test esas filas recibirán la clase mayoritaria (0, 0, 0).

**Defecto 3: filas entrecomilladas.** Los tweets que contenían `"` o `;` quedaron envueltos en comillas dobles escapadas (`""…""`). Al cruzarlos con las etiquetas aparece un patrón muy fuerte:

In [6]:
comillas = train['text'].str.contains('"', regex=False)
print('Filas entrecomilladas en train:', comillas.sum(), '| en test:', test['text'].str.contains('"', regex=False).sum())
display(train[comillas].groupby(LABELS).size().rename('n').to_frame())
print('P(AG=1 | entrecomillada) =', train.loc[comillas, 'AG'].mean().round(3),
      '| P(AG=1 | resto) =', train.loc[~comillas, 'AG'].mean().round(3))
train.loc[comillas, ['text'] + LABELS].head(6)

Filas entrecomilladas en train: 39 | en test: 25


n
HS TR AG    
0  0  1   21
1  0  1   16
   1  1    2

P(AG=1 | entrecomillada) = 1.0 | P(AG=1 | resto) = 0.368


,text,HS,TR,AG
64,"""""T explicas el patriarcado como """"""""simios negratas"""""""" no vengas a exigir nada a las explicaciones de otras cuando se ha explicado desde hace ao...",1,0,1
70,"""""Es bien naco decirle """"""""gabacho"""""""" a los """"""""yunaites""""""""""""",0,0,1
71,"""""Una eurodiputada del BNG desea """"""""violaciones mltiples"""""""" a Casado para que entienda a los inmigrantes https://t.co/pqOHZaE0GU""""",0,0,1
100,"""""Una nueva pelcula de """"""""wini pu"""""""" ??? Necesitamos que las bolsas de cereal traigan cierre ziploc no esas jaladas de osos mayates.""""",0,0,1
115,"""""PSOE-A asegura que en materia de inmigracin el Gobierno de Snchez ha hecho """"""""ms en dos meses que Rajoy en siete aos"""""""" https://t.co/icNBrotefs""""",0,0,1
134,"""""@CNMBrandon Recientemente los medios de comunicacin han difundido las imgenes del """"""""salto"""""""" a la valla de Ceuta (ciudad espaola en frica). A...",0,0,1


Las **39** filas entrecomilladas tienen **AG = 1**, aunque 21 no son discurso de odio (muchas son titulares de prensa). Esto parece un artefacto de anotación o de exportación, no una propiedad del lenguaje. Aun así es una señal legítima de la entrada (la presencia de comillas se lee del propio texto de test), así que se incluye como variable (hipótesis H10) y se discute como limitación en las conclusiones.

## 4. Análisis exploratorio

In [7]:
combo = train[LABELS].astype(str).agg(''.join, axis=1)
resumen = pd.DataFrame({'prevalencia en train': train[LABELS].mean().round(3)})
display(resumen)
display(combo.value_counts().rename_axis('HS-TR-AG').rename('n').to_frame())
print('Casos con TR=1 y HS=0:', int(((train.TR == 1) & (train.HS == 0)).sum()))
print('Casos con AG=1 y HS=0:', int(((train.AG == 1) & (train.HS == 0)).sum()),
      '(de ellos entrecomillados:', int(((train.AG == 1) & (train.HS == 0) & comillas).sum()), ')')

,prevalencia en train
HS,0.437
TR,0.300
AG,0.382


,n
HS-TR-AG,
000,949
111,491
101,145
100,92
110,28
001,26


Casos con TR=1 y HS=0: 0
Casos con AG=1 y HS=0: 26 (de ellos entrecomillados: 21 )


- La distribución de combinaciones está muy concentrada: `000` (55 %), `111` (28 %), `101` (8 %) y `100` (5 %) cubren el 97 % de los datos.
- **TR = 1 implica HS = 1 siempre**: TR es una subcategoría del odio. Esta restricción jerárquica se aplicará a las predicciones.
- AG sin HS es rarísimo (26 casos) y casi todo corresponde a filas entrecomilladas.

In [8]:
train['longitud'] = train['text'].str.len()
display(train.groupby('HS')['longitud'].describe().round(1))

print('Ejemplos por combinación de etiquetas:')
for c, g in train.groupby(combo):
    print(f'\n[{c}] n={len(g)}')
    for s in g.sample(min(3, len(g)), random_state=3)['text']:
        print('   -', s[:140])

,count,mean,std,min,25%,50%,75%,max
HS,,,,,,,,
0,975.0,108.1,57.1,6.0,64.0,98.0,134.5,301.0
1,756.0,99.5,63.0,16.0,51.0,81.0,126.0,306.0


Ejemplos por combinación de etiquetas:

[000] n=949
   - @DrodriguezVen Cllate la jeta mardito feto ambulante. Te vas a morir como una perra hija de la gran puta.
   - @Societatcc Que no se sabe lo de los tirantes copn q no se sabe q puta mana d utilizar a la gente para provocar crispacin parecis seres mezq
   - @YoSoyAntrax el juego de puta madre tio te lodigo y el video la risa te tendria que ensearyoajjaja mentira <3 cuidate bro

[001] n=26
   - ""Video: discriminaron a una nia """"sudaca"""" en el metro de Madrid https://t.co/7fXL71tWfW""
   - ""Mientras el presidente okupa sigue de vacaciones cada da llegan a Espaa cientos de inmigrantes ilegales por el efecto llamada provocado po
   - ""Estoy escuchando el tema """"arabe"""" y la verdad que mi culo se mueve solo""

[100] n=92
   - Un indocumentado...habla y habla y no para y acaba hablando su ignorancia.Empezamos bien. https://t.co/SMRsqjtEBT
   - Los inmigrantes muestran su cara ms violenta con peleas en plena calle  El Pueblo d

### Palabras más asociadas a cada etiqueta
Log-cociente de probabilidades suavizado (presencia del término en positivos vs negativos), sin stopwords y con al menos 8 apariciones. Sirve para ver de un vistazo qué vocabulario separa las clases.

In [9]:
try:
    STOP_ES = set(nltk.corpus.stopwords.words('spanish'))
except LookupError:
    nltk.download('stopwords', quiet=True)
    STOP_ES = set(nltk.corpus.stopwords.words('spanish'))

cv_eda = CountVectorizer(binary=True, min_df=8, token_pattern=r'(?u)\b[a-z]{3,}\b',
                         stop_words=sorted(STOP_ES), lowercase=True)
Xe = cv_eda.fit_transform(train['text'].str.replace(r'https?://\S+|@\w+', ' ', regex=True))
vocab_eda = cv_eda.get_feature_names_out()
tablas = {}
for lab in LABELS:
    y = train[lab].values
    p = (Xe[y == 1].sum(0).A1 + 1) / ((y == 1).sum() + 2)
    q = (Xe[y == 0].sum(0).A1 + 1) / ((y == 0).sum() + 2)
    lo = np.log(p / (1 - p)) - np.log(q / (1 - q))
    tablas[lab] = [f'{vocab_eda[i]} ({lo[i]:+.1f})' for i in np.argsort(-lo)[:15]]
pd.DataFrame(tablas)

,HS,TR,AG
0,putos (+2.8),zorra (+3.0),putos (+3.1)
1,moros (+2.7),perra (+2.4),gorda (+2.8)
2,gorda (+2.6),gorda (+2.4),zorra (+2.4)
3,zorra (+2.4),vieja (+2.2),sevillista (+2.1)
4,sudacas (+2.2),sevillista (+2.0),moros (+2.1)
5,manteros (+2.1),novia (+1.8),jajajajajaja (+1.9)
6,sevillista (+1.9),maldita (+1.7),vieja (+1.9)
7,perra (+1.8),jajajajajaja (+1.7),perra (+1.9)
8,negratas (+1.8),cllate (+1.7),novia (+1.6)
9,trabajo (+1.7),hija (+1.6),sudacas (+1.6)


## 5. Preprocesamiento
La normalización es la **misma para train y test** y se diseñó a partir de los defectos de la sección 3:

| Paso | Por qué |
|---|---|
| `html.unescape` y minúsculas | Hay entidades como `&gt;`; el uso de mayúsculas se guarda aparte como variable |
| URL → `xurl`, mención → `xuser` | Se conserva *que* hay un enlace o una mención (importante para TR) sin memorizar usuarios |
| Quitar `"` y `#` | Las comillas son un artefacto (su presencia se guarda como variable); `#MoroMierdas` → `moromierdas` |
| **Borrar** los caracteres no ASCII | Imita la corrupción del train: en test, un posible `cállate` se convierte en `cllate`, igual que en el entrenamiento |
| Reducir alargamientos (`putaaaa` → `putaa`, `jajajaja` → `jaja`) | Reduce la dispersión del vocabulario |
| **No** quitar stopwords | `tú eres`, `la puta boca`, `hijo de`/`hija de` y `te lo mereces` son claves para las hipótesis H2, H3 y H9 |
| **No** aplicar stemming | El género gramatical (*hijo*/*hija*, *puto*/*puta*) es informativo (H3) y un stemmer lo borraría |

In [10]:
RE_URL  = re.compile(r'https?://\S+|www\.\S+')
RE_MENC = re.compile(r'@\w+')
RE_ALARG = re.compile(r'(.)\1{2,}')

def normalizar(s):
    s = html.unescape(str(s)).lower()
    s = RE_URL.sub(' xurl ', s)
    s = RE_MENC.sub(' xuser ', s)
    s = s.replace('"', ' ').replace('#', ' ')
    s = s.encode('ascii', 'ignore').decode()      # mismo borrado de tildes/ñ que sufrió el train
    s = RE_ALARG.sub(r'\1\1', s)
    return re.sub(r'\s+', ' ', s).strip()

TXT_TR = train['text'].map(normalizar)
TXT_TE = test['text'].map(normalizar)
pd.DataFrame({'original': train['text'].iloc[[0, 5, 13, 66, 300]].values,
              'normalizado': TXT_TR.iloc[[0, 5, 13, 66, 300]].values})

,original,normalizado
0,@SomosRedSaime amigos SAIME mi nombre es walter perez V-13.899.989 wperez001@gmail.com y desde el 16/03/2017 que fui a cita aun no me imprimen el...,xuser amigos saime mi nombre es walter perez v-13.899.989 wperez001 xuser .com y desde el 16/03/2017 que fui a cita aun no me imprimen el pasaport...
1,Gran operacin contra las mafias dedicadas al trfico de inmigrantes https://t.co/ojJon21P51,gran operacin contra las mafias dedicadas al trfico de inmigrantes xurl
2,Antonio Adn: Mi novia es sevillista y es una puta mas | Sevilla 3 Real ... https://t.co/Pk1TYziPU8 va @YouTube,antonio adn: mi novia es sevillista y es una puta mas | sevilla 3 real .. xurl va xuser
3,Minimo denuncia por acoso sexual e intento de violacion telepatica https://t.co/ToVK3NlOtH,minimo denuncia por acoso sexual e intento de violacion telepatica xurl
4,"""""El Partido Popular considera contradictorio el anuncio del PSOE de que retirar las concertinas de Ceuta y Melilla porque """"""""las pusieron ellos""...",el partido popular considera contradictorio el anuncio del psoe de que retirar las concertinas de ceuta y melilla porque las pusieron ellos xurl


## 6. Hipótesis y su verificación
### 6.1 Qué se observó al leer los tweets
La lectura cualitativa de unos 900 tweets del train dejó varias observaciones:

- El corpus corresponde a la tarea **HatEval 2019 (español)**: solo cuenta como odio el que va contra **mujeres** o contra **inmigrantes**. Un insulto durísimo contra un hombre o un político (*"cállate hijo de puta"*) es `000`.
- El corpus se recolectó por **palabras clave** (*puta, perra, zorra, cállate, inmigrante, sudaca, árabe, güero, indocumentado…*), así que los negativos están llenos de esas mismas palabras usadas en otro contexto. La palabra sola no decide: decide el **contexto**.
- La clase `111` es muy formulaica: `@usuaria cállate perra`, `tú eres una zorra`, `te lo mereces por puta`.
- El odio contra inmigrantes casi nunca va dirigido a una persona (TR = 0): son generalizaciones (*"putos moros"*, *"invasión de subsaharianos"*, *"deportación ya"*).
- La clase `100` (odio no agresivo) está dominada por **chistes misóginos** (*"¿Por qué una mujer…?"*) y opiniones antiinmigración en tono "moderado".

### 6.2 Hipótesis
| # | Hipótesis | Cómo se usa en el modelo |
|---|---|---|
| H1 | Los insultos femeninos (*puta, perra, zorra, guarra*) son la señal más fuerte de HS, TR y AG | Conteo de insultos femeninos |
| H2 | *Puta* como **intensificador** (*la puta boca/madre/vez, ni puta idea*) **no** es odio | Patrón de intensificador e interacción insulto sin intensificador |
| H3 | El **género del blanco** decide: *hijo de puta* → no HS; *hija de puta* → HS; los insultos masculinos no son HS; aun con un insulto femenino, la concordancia masculina (*"cállate puta, tú eres **el** traidor"*) indica que el blanco es un hombre | Patrones *hijo*/*hija*, masculino sin femenino y marcas de concordancia del blanco; no se usa stemming |
| H4 | *Cállate* solo indica odio si va acompañado de un insulto femenino | Interacciones *cállate* con y sin insulto femenino |
| H5 | Los insultos y estereotipos de grupo migrante (*sudaca, moro, negrata, subsahariano, ilegal*) indican HS genérico: TR ≈ 0 y AG alto | Conteos de insulto de grupo, *putos + grupo* y vocabulario migratorio |
| H6 | Los encuadres de **invasión/expulsión** (*invasión, plagado, masiva, deportar, a su país*) indican HS agresivo | Patrón de invasión/expulsión |
| H7 | Las palabras clave ambiguas suelen ser benignas: *árabe* (canción de Ecko, comida), *güero*, *acoso/violación* en noticias, *cómemela* | Patrones de contexto benigno |
| H8 | Las generalizaciones y chistes sobre **mujeres** son HS con TR y AG bajos | Patrón *mujer(es)* y chiste × mujer |
| H9 | Con HS = 1, **a quién** se dirige define TR: la mención inicial o el insulto femenino → TR ≈ 1; el tema migrante → TR ≈ 0 | Variables de mención inicial y número de menciones |
| H10 | Las filas entrecomilladas tienen AG = 1 (artefacto) | Variable binaria de comillas |
| H11 | Como se perdieron las tildes y hay alargamientos, los **n-gramas de caracteres** generalizan mejor que las palabras solas | Bloque TF-IDF `char_wb` 2-5 (se mide en la ablación) |

### 6.3 Verificación con datos
Los patrones se escriben sobre el texto normalizado y contemplan las dos grafías (con y sin la vocal acentuada borrada, p. ej. `c(a)?llate`, `imbe?cil`). El mismo diccionario `PATRONES` alimenta esta tabla y las variables del modelo (sección 7).

In [11]:
PATRONES = {
    # nombre: (regex sobre el texto normalizado, hipótesis)
    'insulto_fem':        (r'\b(puta|perra|zorra|guarra|putita|perrita|zorrita)s?\b', 'H1'),
    'puta_intensificador':(r'\b(la|una|de|ni|hasta la|su|que) puta (boca|madre|vez|mierda|idea|polla|gracia|cama|ama)\b'
                           r'|puta madre|que (te|lo|la|me|los) pari', 'H2'),
    'hijo_de_puta':       (r'\bhijos? de (la |tu |su )?(gran |mil |re mil |re )?(puta|perra)', 'H3'),
    'hija_de_puta':       (r'\bhijas? de (la |tu |su )?(gran |mil )?(puta|perra)', 'H3'),
    'insulto_masc':       (r'\b(pendejo|cabro?n|subnormal|cerdo|imbe?cil|gilipollas|marico?n|puto|culiao|perro|mamo?n'
                           r'|rata|hijueputa|hdp|pelotudo|boludo|forro|idiota|est(u)?pido|payaso|ladro?n)\b', 'H3'),
    'blanco_masc':        (r'\b(eres|sos|es) (el|un)\b|\b(maldito|traidor|hijo|wey|bro|hermano|amigo|viejo|tio|man|cabro?n'
                           r'|pendejo|marica|culiao|pelotudo|boludo)\b', 'H3'),
    'blanco_fem':         (r'\b(eres|sos|es) (la|una)\b|\b(maldita|traidora|hija|amiga|vieja|pendeja|gorda|sucia|asquerosa'
                           r'|estu?pida|loca|tonta|barata)\b', 'H3'),
    'callate':            (r'\bc(a)?llate|\bte calles\b|\bcallese\b', 'H4'),
    'trato_directo':      (r'\bt(u)? eres\b|\beres (una|la|mi|un|muy|bien)\b|\bsos (una|la|re|muy|mi)\b|\bvos\b', 'H9'),
    'merecer':            (r'merec', 'H9'),
    'insulto_grupo':      (r'sudac|sudak|\bmoros?\b|moromierd|negrat|\bnegros?\b|manter|panchit|frijoler', 'H5'),
    'putos_grupo':        (r'\bputos? (moros|negros|inmigrantes|sudacas|sudakas|negratas|arabes|rabes|refugiados|indocumentados)', 'H5'),
    'migracion':          (r'inmigra|migrant|subsahar|ilegal|sin papeles|indocumentad|refugiad|\bvalla\b|patera|frontera|extranjer', 'H5'),
    'invasion_expulsion': (r'invas|invad|plagad|masiv|efecto llamada|avalancha|expuls|deport|\bfuera\b|a su pa(i)?s|a sus casa', 'H6'),
    'arabe':              (r'\b(a)?rabes?\b', 'H7'),
    'arabe_benigno':      (r'(tema|cancio?n|ecko|empanad|comida|bail|escuch|falafel|tacos?|lomito|pan|mu?sica|curso|emiratos|coalicio?n)'
                           r'\W+(\w+\W+)?(a)?rabes?\b|\b(a)?rabes?\W+(\w+\W+)?(tema|cancio?n|ecko|empanad|unidos|israel)', 'H7'),
    'guero':              (r'\bg(u)?er[oa]s?\b|\bg(u)?erit[oa]s?\b', 'H7'),
    'noticia_violencia':  (r'acos|violaci|abuso', 'H7'),
    'comemela_semen':     (r'comemel|\bsemen\b', 'H7'),
    'mujer_generica':     (r'\bmujer(es)?\b|\bellas\b|feminaz|feminis|\btodas\b', 'H8'),
    'chiste':             (r'^\W*(-\s*)?(por ?que?\b|que? (es|se|tiene|hace|hacen)\b|en que? se parece|c(o)?mo se dice|cu(a)?ndo)', 'H8'),
    'risa':               (r'jaja|jeje|ajaj|jsjs|haha', '-'),
    'de_mierda':          (r'\bde mierda\b', '-'),
}
RE_PAT = {k: re.compile(p) for k, (p, _) in PATRONES.items()}

def tiene(nombre, textos=TXT_TR):
    return textos.str.contains(RE_PAT[nombre]).values

M = {k: tiene(k) for k in PATRONES}
M['mencion_inicial'] = train['text'].str.match(r'^\W*@').values
M['comillas'] = comillas.values

def evidencia(mascara, hipotesis, condicion, base=None):
    m = np.asarray(mascara, bool) if base is None else (np.asarray(mascara, bool) & base)
    d = train[m]
    return {'Hip.': hipotesis, 'Condición': condicion, 'n': int(m.sum()),
            'P(HS)': d.HS.mean(), 'P(TR)': d.TR.mean(), 'P(AG)': d.AG.mean()}

filas = [
    evidencia(np.ones(len(train)), '-', 'Todo el train (referencia)'),
    evidencia(M['insulto_fem'], 'H1', 'Insulto femenino'),
    evidencia(M['insulto_fem'] & ~M['puta_intensificador'], 'H1', 'Insulto femenino sin intensificador'),
    evidencia(M['insulto_fem'] & M['puta_intensificador'], 'H2', '"puta" como intensificador'),
    evidencia(M['hijo_de_puta'], 'H3', 'hijo(s) de puta/perra'),
    evidencia(M['hija_de_puta'], 'H3', 'hija(s) de puta/perra'),
    evidencia(M['insulto_masc'] & ~M['insulto_fem'], 'H3', 'Insulto masculino sin insulto femenino'),
    evidencia(M['insulto_fem'] & M['blanco_masc'], 'H3', 'Insulto femenino + concordancia masculina (eres el, maldito…)'),
    evidencia(M['insulto_fem'] & M['blanco_fem'], 'H3', 'Insulto femenino + concordancia femenina (eres la, maldita…)'),
    evidencia(M['callate'] & M['insulto_fem'], 'H4', 'cállate + insulto femenino'),
    evidencia(M['callate'] & ~M['insulto_fem'], 'H4', 'cállate sin insulto femenino'),
    evidencia(M['insulto_grupo'], 'H5', 'Insulto/estereotipo de grupo migrante'),
    evidencia(M['putos_grupo'], 'H5', '"putos" + grupo'),
    evidencia(M['migracion'], 'H5', 'Vocabulario migratorio'),
    evidencia(M['invasion_expulsion'], 'H6', 'Invasión / expulsión'),
    evidencia(M['arabe'], 'H7', 'árabe'),
    evidencia(M['arabe'] & M['arabe_benigno'], 'H7', 'árabe en contexto música/comida'),
    evidencia(M['guero'], 'H7', 'güero/a'),
    evidencia(M['noticia_violencia'], 'H7', 'acoso / violación / abuso'),
    evidencia(M['comemela_semen'], 'H7', 'cómemela / semen'),
    evidencia(M['mujer_generica'], 'H8', 'mujer(es), ellas, feminista, todas'),
    evidencia(M['chiste'] & M['mujer_generica'], 'H8', 'Estructura de chiste + mujer'),
    evidencia(M['comillas'], 'H10', 'Fila entrecomillada'),
]
pd.DataFrame(filas).round(2)

,Hip.,Condición,n,P(HS),P(TR),P(AG)
0,-,Todo el train (referencia),1731,0.44,0.30,0.38
1,H1,Insulto femenino,897,0.55,0.53,0.53
2,H1,Insulto femenino sin intensificador,746,0.64,0.62,0.62
3,H2,"""puta"" como intensificador",151,0.08,0.07,0.07
4,H3,hijo(s) de puta/perra,111,0.05,0.01,0.05
5,H3,hija(s) de puta/perra,29,0.69,0.69,0.69
6,H3,Insulto masculino sin insulto femenino,37,0.30,0.11,0.24
7,H3,"Insulto femenino + concordancia masculina (eres el, maldito…)",138,0.14,0.13,0.14
8,H3,"Insulto femenino + concordancia femenina (eres la, maldita…)",186,0.78,0.76,0.76
9,H4,cállate + insulto femenino,375,0.63,0.63,0.63


**H9: TR condicionado a que haya odio.** Si solo se miran los tweets con HS = 1, ¿qué separa el odio dirigido (TR = 1) del genérico?

In [12]:
hs1 = (train.HS == 1).values
filas_h9 = [
    evidencia(np.ones(len(train), bool), 'H9', 'Todos los HS=1 (referencia)', hs1),
    evidencia(M['mencion_inicial'], 'H9', 'Empieza con @mención', hs1),
    evidencia(~M['mencion_inicial'], 'H9', 'No empieza con @mención', hs1),
    evidencia(M['insulto_fem'], 'H9', 'Con insulto femenino', hs1),
    evidencia(M['trato_directo'], 'H9', 'Trato directo (tú eres / sos / vos)', hs1),
    evidencia(M['migracion'] | M['insulto_grupo'], 'H9', 'Tema migrante', hs1),
    evidencia(M['mujer_generica'] & ~M['insulto_fem'], 'H9', 'Mujeres en genérico, sin insulto', hs1),
]
pd.DataFrame(filas_h9).drop(columns='P(HS)').round(2)

,Hip.,Condición,n,P(TR),P(AG)
0,H9,Todos los HS=1 (referencia),756,0.69,0.84
1,H9,Empieza con @mención,361,0.94,0.93
2,H9,No empieza con @mención,395,0.46,0.76
3,H9,Con insulto femenino,493,0.96,0.96
4,H9,Trato directo (tú eres / sos / vos),140,0.98,0.93
5,H9,Tema migrante,167,0.13,0.72
6,H9,"Mujeres en genérico, sin insulto",38,0.08,0.32


**Lectura de la evidencia.**
- **H1 y H2 se confirman y se complementan**: el insulto femenino sube P(HS) de 0,44 a 0,55, y a **0,65** si se descartan los usos como intensificador. Cuando *puta* es intensificador, P(HS) cae a **0,07**.
- **H3 se confirma con fuerza**: *hijo de puta* tiene P(HS) ≈ 0,05 y *hija de puta* ≈ 0,69. Incluso con un insulto femenino, si el resto del tweet concuerda en masculino (*eres el*, *maldito*, *wey*) P(HS) cae a ≈ 0,14, y si concuerda en femenino sube a ≈ 0,78. Por eso el modelo no usa stemming. (Las marcas de concordancia se agregaron después de ver los primeros errores del modelo, sección 9.)
- **H4 se confirma**: *cállate* sin insulto femenino tiene P(HS) ≈ 0,10, y con insulto ≈ 0,63.
- **H5 y H6 se confirman**: los insultos de grupo migrante dan P(HS) muy alta con P(TR) casi nula; *putos + grupo* es 100 % odio agresivo.
- **H7 se confirma**: *árabe* tiene P(HS) ≈ 0,19 en general y ≈ 0,04 en contexto de música o comida.
- **H8 se confirma**: el odio hacia las mujeres en genérico tiene TR y AG bajos.
- **H9 se confirma**: con HS = 1, empezar con una mención o llevar un insulto femenino lleva P(TR) a ≈ 0,95, y el tema migrante la baja a ≈ 0,14.
- **H10** es un artefacto, pero perfectamente consistente (39/39).
- **H11** se mide en la ablación de la sección 8.

## 7. Representación del texto
Tres bloques que se concatenan:

1. **TF-IDF de palabras 1-2** (`sublinear_tf`, `min_df=2`, incluye tokens de una letra para no perder `t eres`). Captura bigramas como `hija de`, `puta boca`, `xuser callate`.
2. **TF-IDF de caracteres `char_wb` 2-5** (H11): robusto a las dos grafías (`cllate`/`callate`), a los alargamientos y a las variantes (`zorr`, `perr`, `moromierd`).
3. **Bloque de léxico y estructura** (H1-H10): un conteo por patrón, las interacciones que la sección 6 mostró decisivas y variables de estructura (menciones, URL, hashtags, comillas, mayúsculas, longitud, signos). Se estandariza y se pondera con `PESO_LEX` para que no domine a los bloques dispersos.

In [13]:
def variables_lexicas(raw, norm):
    c = {k: len(RE_PAT[k].findall(norm)) for k in PATRONES}
    f = list(c.values())
    f += [
        int(c['insulto_fem'] > 0 and c['puta_intensificador'] == 0),   # H1 sin H2
        int(c['callate'] > 0 and c['insulto_fem'] > 0),                # H4
        int(c['callate'] > 0 and c['insulto_fem'] == 0),               # H4
        int(c['insulto_masc'] > 0 and c['insulto_fem'] == 0),          # H3
        int(c['chiste'] > 0 and c['mujer_generica'] > 0),              # H8
        int(c['arabe'] > 0 and c['arabe_benigno'] == 0),               # H7
        int(bool(re.match(r'^\W*@', raw))),                            # H9 mención inicial
        raw.count('@'),                                                # H9
        int('http' in raw),
        raw.count('#'),
        int('"' in raw),                                               # H10
        sum(ch.isupper() for ch in raw) / max(1, sum(ch.isalpha() for ch in raw)),
        np.log1p(len(raw)),
        raw.count('!'),
        raw.count('?'),
    ]
    return f

LEX_NOMBRES = list(PATRONES) + ['fem_sin_intensificador', 'callate_con_fem', 'callate_sin_fem', 'masc_sin_fem',
                                'chiste_mujer', 'arabe_no_benigno', 'mencion_inicial', 'n_menciones', 'tiene_url',
                                'n_hashtags', 'comillas', 'prop_mayusculas', 'log_longitud', 'n_exclamaciones',
                                'n_preguntas']
L_TR = np.array([variables_lexicas(r, n) for r, n in zip(train['text'], TXT_TR)], dtype=float)
L_TE = np.array([variables_lexicas(r, n) for r, n in zip(test['text'], TXT_TE)], dtype=float)
N_LEX = L_TR.shape[1]
assert N_LEX == len(LEX_NOMBRES)
print('Variables de léxico:', N_LEX)

def ajustar_bloques(txt_tr, txt_te, L_tr, L_te):
    """Ajusta los tres bloques SOLO con txt_tr / L_tr y transforma ambos conjuntos."""
    vw, vc, sc = TfidfVectorizer(**CFG_WORD), TfidfVectorizer(**CFG_CHAR), StandardScaler()
    B_tr = {'word': vw.fit_transform(txt_tr), 'char': vc.fit_transform(txt_tr),
            'lex': sparse.csr_matrix(sc.fit_transform(L_tr) * PESO_LEX)}
    B_te = {'word': vw.transform(txt_te), 'char': vc.transform(txt_te),
            'lex': sparse.csr_matrix(sc.transform(L_te) * PESO_LEX)}
    return B_tr, B_te, (vw, vc, sc)

def unir(B, bloques):
    return sparse.hstack([B[b] for b in bloques]).tocsr()

TODOS = ['word', 'char', 'lex']

Variables de léxico: 38


## 8. Modelos y experimentos
### 8.1 Protocolo
- **Validación cruzada de 5 particiones**, estratificada por la combinación HS-TR-AG, con **las mismas particiones para todos los modelos**. Así las predicciones fuera de muestra (OOF) son comparables y se pueden ensamblar.
- En cada partición los vectorizadores y el escalador se ajustan **solo con los 4/5 de entrenamiento**.
- Métrica principal, **`F1 medio`**: F1 de la clase positiva de cada etiqueta, promediado sobre HS, TR y AG. Equivale a `f1_score(Y, Yp, average='macro')` de scikit-learn sobre la matriz multietiqueta, la implementación más común de «F1 promedio por etiqueta».
- También se reportan el F1 micro, el **F1-macro de dos clases** por etiqueta (promedia la clase 0 y la 1, como en HatEval; siempre da más alto) y el EMR (*exact match ratio*: acierto simultáneo de las tres etiquetas).
- Toda predicción aplica la restricción jerárquica **TR := TR · HS**.

> **Lección de la primera submission.** La v1 se eligió y se reportó con el F1-macro de dos clases (0,847 en CV), pero Kaggle dio **0,78516**. Al recalcular la CV con las distintas definiciones de F1, el F1 de la clase positiva da 0,809, y con un leaderboard público de ~577 filas su rango de ruido (percentiles 5-95) va de 0,781 a 0,831. El puntaje de Kaggle cae dentro de ese rango, mientras que con el F1-macro de dos clases (rango 0,827-0,865) quedaría fuera. Por eso la métrica principal de esta versión es el F1 de la clase positiva.

### 8.2 Modelos
| Modelo | Descripción |
|---|---|
| **CNB** | Complement Naive Bayes sobre los TF-IDF (baseline clásico de texto) |
| **LR** | Regresión logística binaria por etiqueta, `class_weight='balanced'` |
| **SVM** | SVM lineal por etiqueta, calibrado con sigmoide (Platt) para obtener probabilidades |
| **NB-LR** | Regresión logística sobre los n-gramas reponderados con el log-cociente de Naive Bayes (Wang & Manning, 2012) |
| **Powerset** | Regresión logística multiclase sobre las combinaciones HS-TR-AG; las probabilidades de cada etiqueta salen de sumar las combinaciones que la contienen. Aprovecha la estructura conjunta (TR ⊂ HS, chistes = `100`, etc.) |

In [14]:
class NBLogReg(BaseEstimator, ClassifierMixin):
    """Regresión logística sobre n-gramas reponderados por el log-cociente de Naive Bayes.
    Las últimas n_lex columnas (bloque de léxico) no se reponderan."""
    def __init__(self, alpha=1.0, C=4.0, n_lex=0):
        self.alpha, self.C, self.n_lex = alpha, C, n_lex

    def fit(self, X, y):
        d = X.shape[1] - self.n_lex
        Xb = (X[:, :d] > 0).astype(float)
        p = self.alpha + Xb[y == 1].sum(0).A1
        q = self.alpha + Xb[y == 0].sum(0).A1
        r = np.log((p / p.sum()) / (q / q.sum()))
        self.r_ = np.concatenate([r, np.ones(self.n_lex)])
        self.clf_ = LogisticRegression(C=self.C, max_iter=5000, class_weight='balanced')
        self.clf_.fit(X.multiply(self.r_).tocsr(), y)
        self.classes_ = self.clf_.classes_
        return self

    def predict_proba(self, X):
        return self.clf_.predict_proba(X.multiply(self.r_).tocsr())

MODELOS = {
    'CNB':   lambda: ComplementNB(alpha=0.3),
    'LR':    lambda: LogisticRegression(C=C_LR, max_iter=5000, class_weight='balanced'),
    'SVM':   lambda: CalibratedClassifierCV(LinearSVC(C=C_SVM, class_weight='balanced'), cv=3, method='sigmoid'),
    'NB-LR': lambda: NBLogReg(alpha=NB_ALPHA, C=C_LR, n_lex=N_LEX),
}
COMBO = combo.values

def marginales(P_combo, clases):
    """P(etiqueta=1) = suma de las probabilidades de las combinaciones que la contienen."""
    return np.column_stack([P_combo[:, [k for k, c in enumerate(clases) if c[j] == '1']].sum(1)
                            for j in range(len(LABELS))])

def predecir(nombre, Xtr, Ytr, combo_tr, Xte):
    """Entrena el modelo `nombre` y devuelve una matriz (n_te, 3) con P(HS), P(TR), P(AG)."""
    if nombre == 'Powerset':
        m = LogisticRegression(C=C_LR, max_iter=5000).fit(Xtr, combo_tr)
        return marginales(m.predict_proba(Xte), m.classes_)
    return np.column_stack([MODELOS[nombre]().fit(Xtr, Ytr[:, j]).predict_proba(Xte)[:, 1]
                            for j in range(len(LABELS))])

def binarizar(P, umbrales=(0.5, 0.5, 0.5)):
    Yp = (P >= np.asarray(umbrales)).astype(int)
    Yp[:, 1] *= Yp[:, 0]                       # restricción jerárquica TR ⊂ HS
    return Yp

def metricas(Yp, nombre, y_true=None):
    yt = Y if y_true is None else y_true
    f1b = [f1_score(yt[:, j], Yp[:, j]) for j in range(3)]                      # clase positiva
    f1m = [f1_score(yt[:, j], Yp[:, j], average='macro') for j in range(3)]     # promedio de las 2 clases
    return {'Modelo': nombre, **{f'F1 {l}': v for l, v in zip(LABELS, f1b)},
            'F1 medio': np.mean(f1b), 'F1 micro': f1_score(yt, Yp, average='micro'),
            'F1-macro 2 clases': np.mean(f1m), 'EMR': (Yp == yt).all(1).mean()}

In [15]:
if VALIDAR:
    FOLDS = list(StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED).split(TXT_TR, COMBO))
    FOLD_DATA = []
    for tr, te in FOLDS:            # los bloques se ajustan una sola vez por partición y se reutilizan
        B_tr, B_te, _ = ajustar_bloques(TXT_TR.iloc[tr], TXT_TR.iloc[te], L_TR[tr], L_TR[te])
        FOLD_DATA.append((tr, te, B_tr, B_te))
    log('Bloques por partición listos. Dimensiones de la 1.ª partición:',
        {k: v.shape[1] for k, v in FOLD_DATA[0][2].items()})

    P_OOF = {}
    def oof(nombre, bloques, clave=None):
        clave = clave or nombre
        P = np.zeros((len(train), len(LABELS)))
        for tr, te, B_tr, B_te in FOLD_DATA:
            P[te] = predecir(nombre, unir(B_tr, bloques), Y[tr], COMBO[tr], unir(B_te, bloques))
        P_OOF[clave] = P
        log('OOF listo:', clave)
        return P

[     1s] Bloques por partición listos. Dimensiones de la 1.ª partición: {'word': 3538, 'char': 16488, 'lex': 38}


### 8.3 Ablación de la representación
Mismo modelo (LR) y umbral 0,5, agregando un bloque a la vez. Esto mide el aporte de los n-gramas de caracteres (H11) y del bloque de léxico construido con las hipótesis H1-H10.

In [16]:
if VALIDAR:
    ablacion = []
    for clave, bloques in [('LR · palabras', ['word']), ('LR · caracteres', ['char']),
                           ('LR · palabras + caracteres', ['word', 'char']),
                           ('LR · palabras + caracteres + léxico', TODOS)]:
        ablacion.append(metricas(binarizar(oof('LR', bloques, clave)), clave))
    P_OOF['LR'] = P_OOF['LR · palabras + caracteres + léxico']
    display(pd.DataFrame(ablacion).set_index('Modelo').round(3))

[     2s] OOF listo: LR · palabras


[     2s] OOF listo: LR · caracteres


[     3s] OOF listo: LR · palabras + caracteres


[     6s] OOF listo: LR · palabras + caracteres + léxico


,F1 HS,F1 TR,F1 AG,F1 medio,F1 micro,F1-macro 2 clases,EMR
Modelo,,,,,,,
LR · palabras,0.754,0.817,0.741,0.771,0.767,0.816,0.713
LR · caracteres,0.759,0.785,0.726,0.757,0.754,0.806,0.698
LR · palabras + caracteres,0.777,0.817,0.749,0.781,0.778,0.826,0.723
LR · palabras + caracteres + léxico,0.793,0.823,0.791,0.803,0.801,0.840,0.755


### 8.4 Comparación de modelos
Todos con la representación completa (palabras + caracteres + léxico), salvo CNB, que necesita variables no negativas y usa solo los TF-IDF.

In [17]:
if VALIDAR:
    oof('CNB', ['word', 'char'])
    for nombre in ['SVM', 'NB-LR', 'Powerset']:
        oof(nombre, TODOS)
    comparacion = [metricas(binarizar(P_OOF[k]), k) for k in ['CNB', 'LR', 'SVM', 'NB-LR', 'Powerset']]
    display(pd.DataFrame(comparacion).set_index('Modelo').round(3))

[     6s] OOF listo: CNB


[     7s] OOF listo: SVM


[     9s] OOF listo: NB-LR


[    12s] OOF listo: Powerset


,F1 HS,F1 TR,F1 AG,F1 medio,F1 micro,F1-macro 2 clases,EMR
Modelo,,,,,,,
CNB,0.759,0.780,0.709,0.749,0.748,0.805,0.716
LR,0.793,0.823,0.791,0.803,0.801,0.840,0.755
SVM,0.791,0.830,0.788,0.803,0.801,0.842,0.756
NB-LR,0.796,0.821,0.787,0.801,0.800,0.842,0.760
Powerset,0.796,0.835,0.788,0.806,0.804,0.846,0.780


### 8.5 Ensamble
Se promedian las probabilidades de varios modelos. Para no sobreajustar la selección a las predicciones OOF, solo se evalúa un puñado de combinaciones predefinidas, y se elige la de mayor `F1 medio` con umbral 0,5.

In [18]:
if VALIDAR:
    CANDIDATOS = [['LR'], ['Powerset'], ['LR', 'Powerset'], ['LR', 'SVM'], ['LR', 'Powerset', 'SVM'],
                  ['LR', 'Powerset', 'NB-LR'], ['LR', 'Powerset', 'SVM', 'NB-LR']]
    res_ens = []
    for cand in CANDIDATOS:
        P = np.mean([P_OOF[k] for k in cand], axis=0)
        res_ens.append(metricas(binarizar(P), ' + '.join(cand)))
    res_ens = pd.DataFrame(res_ens).set_index('Modelo')
    display(res_ens.round(3))
    ENSAMBLE = CANDIDATOS[int(np.argmax(res_ens['F1 medio'].values))]
    P_ENS = np.mean([P_OOF[k] for k in ENSAMBLE], axis=0)
    log('Ensamble elegido:', ENSAMBLE)
else:
    ENSAMBLE = ENSAMBLE_POR_DEFECTO

,F1 HS,F1 TR,F1 AG,F1 medio,F1 micro,F1-macro 2 clases,EMR
Modelo,,,,,,,
LR,0.793,0.823,0.791,0.803,0.801,0.840,0.755
Powerset,0.796,0.835,0.788,0.806,0.804,0.846,0.780
LR + Powerset,0.798,0.836,0.791,0.809,0.806,0.847,0.769
LR + SVM,0.790,0.827,0.792,0.803,0.801,0.841,0.754
LR + Powerset + SVM,0.797,0.834,0.794,0.808,0.806,0.846,0.767
LR + Powerset + NB-LR,0.798,0.833,0.793,0.808,0.806,0.846,0.769
LR + Powerset + SVM + NB-LR,0.797,0.835,0.790,0.807,0.805,0.846,0.767


[    12s] Ensamble elegido: ['LR', 'Powerset']


### 8.6 Umbrales por etiqueta (validación cruzada anidada)
Elegir el umbral sobre las mismas predicciones con que se evalúa infla el resultado. Por eso, para estimar honestamente el efecto, en cada partición el umbral se elige con las **otras 4** particiones y se aplica a la que quedó fuera. Los umbrales finales solo se adoptan si esta estimación honesta supera al umbral fijo de 0,5.

In [19]:
def umbrales_optimos(P, idx):
    """Umbral por etiqueta que maximiza el F1 de la clase positiva en las filas idx."""
    return np.array([max(UMBRAL_GRID, key=lambda t: f1_score(Y[idx, j], (P[idx, j] >= t).astype(int)))
                     for j in range(len(LABELS))])

def umbrales_anidados(P):
    """Predicciones con umbrales elegidos en las otras particiones (estimación honesta)."""
    Yp = np.zeros_like(Y)
    for tr, te in FOLDS:
        Yp[te] = binarizar(P[te], umbrales_optimos(P, tr))
    return Yp

if VALIDAR:
    base_05   = metricas(binarizar(P_ENS), 'Ensamble · umbral 0,5')
    anidado   = metricas(umbrales_anidados(P_ENS), 'Ensamble · umbral por etiqueta (CV anidada)')
    u_todos   = umbrales_optimos(P_ENS, np.arange(len(train)))
    optimista = metricas(binarizar(P_ENS, u_todos), 'Ensamble · umbral elegido sobre todo el OOF (optimista)')
    display(pd.DataFrame([base_05, anidado, optimista]).set_index('Modelo').round(3))
    usar_u = anidado['F1 medio'] > base_05['F1 medio']
    UMBRALES = u_todos if usar_u else np.array([0.5, 0.5, 0.5])
    RESULTADO_ENS = {**(anidado if usar_u else base_05), 'Modelo': 'Ensamble ' + ' + '.join(ENSAMBLE)}
else:
    UMBRALES = np.array([0.5, 0.5, 0.5])
print('Umbrales del ensamble:', dict(zip(LABELS, UMBRALES)))

,F1 HS,F1 TR,F1 AG,F1 medio,F1 micro,F1-macro 2 clases,EMR
Modelo,,,,,,,
"Ensamble · umbral 0,5",0.798,0.836,0.791,0.809,0.806,0.847,0.769
Ensamble · umbral por etiqueta (CV anidada),0.793,0.832,0.793,0.806,0.803,0.842,0.754
Ensamble · umbral elegido sobre todo el OOF (optimista),0.799,0.836,0.797,0.811,0.808,0.846,0.760


Umbrales del ensamble: {'HS': np.float64(0.5), 'TR': np.float64(0.5), 'AG': np.float64(0.5)}


### 8.7 Stacking
En vez de promediar, un **metamodelo** por etiqueta (regresión logística) aprende a combinar las 15 probabilidades fuera de muestra de los cinco modelos (3 etiquetas × 5 modelos). Así puede aprender, por ejemplo, cuánto confiar en cada modelo y usar P(TR) o P(AG) como evidencia para HS.

El metamodelo se evalúa con **otra** validación cruzada de 5 particiones (semilla distinta) sobre las probabilidades OOF, así que nunca predice una fila que usó para entrenarse. Queda una optimización leve, propia del stacking: los modelos base de cada partición vieron filas que el metamodelo usa como prueba. Por eso solo se adopta si mejora al ensamble con claridad.

In [20]:
BASES_STACK = ['CNB', 'LR', 'SVM', 'NB-LR', 'Powerset']
BLOQUES_BASE = {'CNB': ['word', 'char']}            # el resto usa los tres bloques
MARGEN_STACK = 0.003                                 # mejora mínima de F1 medio para adoptarlo

def meta_modelo():
    return LogisticRegression(C=1.0, max_iter=3000, class_weight='balanced')

if VALIDAR:
    Z_OOF = np.hstack([P_OOF[k] for k in BASES_STACK])
    P_STACK = np.zeros_like(P_ENS)
    for tr, te in StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED + 1).split(Z_OOF, COMBO):
        for j in range(len(LABELS)):
            P_STACK[te, j] = meta_modelo().fit(Z_OOF[tr], Y[tr, j]).predict_proba(Z_OOF[te])[:, 1]
    RESULTADO_STACK = metricas(binarizar(P_STACK), 'Stacking (metamodelo LR sobre 5 modelos)')
    display(pd.DataFrame([RESULTADO_ENS, RESULTADO_STACK]).set_index('Modelo').round(3))

    if RESULTADO_STACK['F1 medio'] > RESULTADO_ENS['F1 medio'] + MARGEN_STACK:
        FINAL, P_FINAL_OOF, UMBRALES_FINAL, RESULTADO_FINAL_CV = 'stacking', P_STACK, np.array([0.5] * 3), RESULTADO_STACK
    else:
        FINAL, P_FINAL_OOF, UMBRALES_FINAL, RESULTADO_FINAL_CV = 'ensamble', P_ENS, UMBRALES, RESULTADO_ENS
else:
    FINAL, UMBRALES_FINAL = 'ensamble', UMBRALES
log('Modelo final:', FINAL)

,F1 HS,F1 TR,F1 AG,F1 medio,F1 micro,F1-macro 2 clases,EMR
Modelo,,,,,,,
Ensamble LR + Powerset,0.798,0.836,0.791,0.809,0.806,0.847,0.769
Stacking (metamodelo LR sobre 5 modelos),0.805,0.839,0.802,0.815,0.813,0.850,0.775


[    12s] Modelo final: stacking


### 8.8 Sensibilidad a los hiperparámetros
Se repite la validación cruzada del ensamble elegido variando un hiperparámetro a la vez. Si el resultado cambia poco, la configuración está en una zona estable y no depende de un ajuste fino que podría no transferirse al test.

In [21]:
def cv_config(c_lr, peso, word_ng, char_ng):
    """F1 OOF del ensamble con otra configuración; restaura los valores globales al terminar."""
    global C_LR, PESO_LEX, CFG_WORD, CFG_CHAR
    original = (C_LR, PESO_LEX, CFG_WORD, CFG_CHAR)
    try:
        C_LR, PESO_LEX = c_lr, peso
        CFG_WORD, CFG_CHAR = dict(CFG_WORD, ngram_range=word_ng), dict(CFG_CHAR, ngram_range=char_ng)
        P = np.zeros((len(train), len(LABELS)))
        for tr, te in FOLDS:
            B_tr, B_te, _ = ajustar_bloques(TXT_TR.iloc[tr], TXT_TR.iloc[te], L_TR[tr], L_TR[te])
            Xa, Xb = unir(B_tr, TODOS), unir(B_te, TODOS)
            P[te] = np.mean([predecir(k, Xa, Y[tr], COMBO[tr], Xb) for k in ENSAMBLE], axis=0)
        return metricas(binarizar(P), f'C={c_lr} · peso léxico={peso} · palabras {word_ng} · caracteres {char_ng}')
    finally:
        C_LR, PESO_LEX, CFG_WORD, CFG_CHAR = original

if VALIDAR:
    w0, c0 = CFG_WORD['ngram_range'], CFG_CHAR['ngram_range']
    configs = [(C_LR, PESO_LEX, w0, c0), (2.0, PESO_LEX, w0, c0), (8.0, PESO_LEX, w0, c0),
               (C_LR, 0.15, w0, c0), (C_LR, 1.0, w0, c0), (C_LR, PESO_LEX, (1, 3), c0), (C_LR, PESO_LEX, w0, (1, 6))]
    sensibilidad = pd.DataFrame([cv_config(*cfg) for cfg in configs]).set_index('Modelo')
    log('Sensibilidad lista')
    display(sensibilidad.round(3))
    print('Rango del F1 medio:', sensibilidad['F1 medio'].min().round(3), '–',
          sensibilidad['F1 medio'].max().round(3))

[    58s] Sensibilidad lista


,F1 HS,F1 TR,F1 AG,F1 medio,F1 micro,F1-macro 2 clases,EMR
Modelo,,,,,,,
"C=4.0 · peso léxico=0.3 · palabras (1, 2) · caracteres (2, 5)",0.798,0.836,0.791,0.809,0.806,0.847,0.769
"C=2.0 · peso léxico=0.3 · palabras (1, 2) · caracteres (2, 5)",0.796,0.831,0.794,0.807,0.805,0.845,0.769
"C=8.0 · peso léxico=0.3 · palabras (1, 2) · caracteres (2, 5)",0.798,0.838,0.784,0.807,0.804,0.845,0.765
"C=4.0 · peso léxico=0.15 · palabras (1, 2) · caracteres (2, 5)",0.794,0.833,0.791,0.806,0.804,0.846,0.771
"C=4.0 · peso léxico=1.0 · palabras (1, 2) · caracteres (2, 5)",0.798,0.837,0.794,0.810,0.807,0.847,0.771
"C=4.0 · peso léxico=0.3 · palabras (1, 3) · caracteres (2, 5)",0.796,0.834,0.792,0.808,0.805,0.846,0.768
"C=4.0 · peso léxico=0.3 · palabras (1, 2) · caracteres (1, 6)",0.797,0.836,0.792,0.809,0.806,0.847,0.770


Rango del F1 medio: 0.806 – 0.81


### 8.9 Resumen de experimentos

In [22]:
if VALIDAR:
    resumen_exp = pd.DataFrame(ablacion + comparacion[:1] + comparacion[2:] +
                               [RESULTADO_ENS, RESULTADO_STACK,
                                {**RESULTADO_FINAL_CV, 'Modelo': f'Modelo final ({FINAL})'}])
    display(resumen_exp.set_index('Modelo').round(3))

,F1 HS,F1 TR,F1 AG,F1 medio,F1 micro,F1-macro 2 clases,EMR
Modelo,,,,,,,
LR · palabras,0.754,0.817,0.741,0.771,0.767,0.816,0.713
LR · caracteres,0.759,0.785,0.726,0.757,0.754,0.806,0.698
LR · palabras + caracteres,0.777,0.817,0.749,0.781,0.778,0.826,0.723
LR · palabras + caracteres + léxico,0.793,0.823,0.791,0.803,0.801,0.840,0.755
CNB,0.759,0.780,0.709,0.749,0.748,0.805,0.716
SVM,0.791,0.830,0.788,0.803,0.801,0.842,0.756
NB-LR,0.796,0.821,0.787,0.801,0.800,0.842,0.760
Powerset,0.796,0.835,0.788,0.806,0.804,0.846,0.780
Ensamble LR + Powerset,0.798,0.836,0.791,0.809,0.806,0.847,0.769


## 9. Análisis de errores
### 9.1 Matrices de confusión (predicciones OOF del modelo final)

In [23]:
if VALIDAR:
    Yp_final_oof = binarizar(P_FINAL_OOF, UMBRALES_FINAL)
    for j, lab in enumerate(LABELS):
        cm = confusion_matrix(Y[:, j], Yp_final_oof[:, j])
        display(pd.DataFrame(cm, index=[f'{lab} real=0', f'{lab} real=1'],
                             columns=[f'pred=0', f'pred=1']))

,pred=0,pred=1
HS real=0,835,140
HS real=1,153,603


,pred=0,pred=1
TR real=0,1118,94
TR real=1,76,443


,pred=0,pred=1
AG real=0,923,146
AG real=1,121,541


### 9.2 Variables con más peso
Regresión logística binaria entrenada con todo el train sobre la representación completa. Los n-gramas de caracteres se muestran entre corchetes y las variables de léxico con el prefijo `LEX:`.

In [24]:
B_full, _, (vw_f, vc_f, _) = ajustar_bloques(TXT_TR, TXT_TR.iloc[:1], L_TR, L_TR[:1])
X_full = unir(B_full, TODOS)
nombres = np.concatenate([vw_f.get_feature_names_out(),
                          [f'[{g}]' for g in vc_f.get_feature_names_out()],
                          [f'LEX:{n}' for n in LEX_NOMBRES]])
top = {}
for j, lab in enumerate(LABELS):
    coef = MODELOS['LR']().fit(X_full, Y[:, j]).coef_[0]
    top[f'{lab} (+)'] = nombres[np.argsort(-coef)[:20]]
    top[f'{lab} (−)'] = nombres[np.argsort(coef)[:20]]
pd.DataFrame(top)

,HS (+),HS (−),TR (+),TR (−),AG (+),AG (−)
0,LEX:insulto_grupo,la puta,LEX:fem_sin_intensificador,la puta,LEX:comillas,la puta
1,zorra,mi perra,sudaca,mi perra,LEX:fem_sin_intensificador,mi perra
2,LEX:migracion,LEX:hijo_de_puta,LEX:insulto_fem,LEX:hijo_de_puta,LEX:insulto_grupo,LEX:hijo_de_puta
3,al,LEX:puta_intensificador,mi polla,a la,al,de la
4,perra,ac,LEX:blanco_fem,son,LEX:blanco_fem,y yo
5,LEX:insulto_fem,mujer es,zorra,todo,LEX:insulto_fem,hijo
6,LEX:blanco_fem,la,perra,quien,zorra,ac
7,callate puta,en,sirvienta,en,callate puta,una
8,se,xuser xurl,callate puta,puta rata,nos,la
9,sirvienta,hijo,y la,a mi,mi polla,a mi


### 9.3 Ejemplos de errores

In [25]:
if VALIDAR:
    for j, lab in enumerate(LABELS):
        fp = np.where((Yp_final_oof[:, j] == 1) & (Y[:, j] == 0))[0]
        fn = np.where((Yp_final_oof[:, j] == 0) & (Y[:, j] == 1))[0]
        print(f'===== {lab}: {len(fp)} falsos positivos, {len(fn)} falsos negativos')
        for tipo, idx in [('FP', fp[np.argsort(-P_FINAL_OOF[fp, j])][:5]),
                          ('FN', fn[np.argsort(P_FINAL_OOF[fn, j])][:5])]:
            for i in idx:
                print(f'  {tipo} p={P_FINAL_OOF[i, j]:.2f} [{COMBO[i]}]  {train.text.iloc[i][:130]}')

===== HS: 140 falsos positivos, 153 falsos negativos
  FP p=0.97 [000]  Los asaltantes subsaharianos de la valla de ceuta estan bien entrenados militarmente. Acaso en las fotos ve a muchas mujeres y nio
  FP p=0.96 [000]  Casado y Rivera contra la inmigracin ilegal. Snchez e Iglesias a favor de la invasin masiva y violenta de subsaharianos.A quin pre
  FP p=0.95 [000]  Y CHILEEE??? CUNDO VISITAN A LOS SUDACAS  https://t.co/OP6Tbw2mBk
  FP p=0.95 [000]  @defays_mikael @rickcenturion cllate zorra ac los leprosos no opinan
  FP p=0.95 [000]  Racismo es que hagas enfrijoladas con frijoles geros y no negros.
  FN p=0.06 [101]  ""los estupidos """"rabes"""" de la universidad https://t.co/QhxBVSRmMx""
  FN p=0.09 [101]  Indios hijos de puta. Soltad al gabacho
  FN p=0.10 [101]  Mami puso una emisora con cancin rabe y siento que en cualquier momento vamos a explotar.
  FN p=0.10 [111]  Comemela XDDD y la cara tambin!!   No nos pasemos  https://t.co/t3o5yHY0zb
  FN p=0.10 [100]  Pues si la verd

**Patrones en los errores** (con lo observado en la lectura y en las salidas de arriba):
- **Ruido de anotación.** *perra/zorra* en tono afectuoso se etiqueta de forma inconsistente: *"Cllate todo te queda bien zorra te amo"* es `000`, pero *"Jeje callate perra"* y *"Tú eres bien perra"* son `110`. Igual pasa con *cómemela*: casi siempre es `000` (H7), pero *"Comemela mientras te meto los dedos"* es `111`. Ningún modelo de bolsa de palabras puede resolver esa inconsistencia.
- **El blanco no es una mujer.** En *"cllate zorra acá los leprosos no opinan"* (`000`) el plural masculino indica que el blanco no es una mujer, y en *"Cllate que asustas a la perra"* (`000`) *perra* es el animal. El modelo sigue dando P ≈ 0,95-0,97: las marcas de concordancia (H3) mejoran el F1 en conjunto, pero una sola palabra no alcanza a contrarrestar la fórmula *cállate + puta/zorra/perra*.
- **La frontera entre opinión y odio xenófobo.** *"Los asaltantes subsaharianos de la valla de Ceuta están bien entrenados militarmente…"* o *"…a favor de la invasión masiva y violenta de subsaharianos"* están anotados como `000`, aunque comparten casi todo el vocabulario de la clase `101`.
- **Estereotipos implícitos y blancos de grupo con fórmulas masculinas.** Los falsos negativos de HS incluyen estereotipos sin palabras clave (*"Mami puso una emisora con canción árabe y siento que en cualquier momento vamos a explotar"*) y odio de grupo escrito con fórmulas que en el resto del corpus son benignas (*"Indios hijos de puta"*, donde *hijos de puta* empuja hacia el no-odio por H3).
- **TR es la etiqueta más fácil** porque, una vez detectado el odio, la presencia de un blanco individual (mención o insulto femenino) es muy explícita (H9). **AG es la más difícil**: la frontera entre odio agresivo y no agresivo es sutil (por ejemplo, opinión antiinmigración "moderada" frente a llamado a la expulsión).

## 10. Modelo final y archivo de envío
Se reentrenan los modelos con **todo el train**: los bloques se ajustan solo con el train y el test únicamente se transforma.
- Si el modelo final es el **ensamble**, se promedian las probabilidades de sus miembros y se aplican los umbrales de la sección 8.6.
- Si es el **stacking**, los cinco modelos base predicen el test y el metamodelo (entrenado con las probabilidades OOF del train) combina esas predicciones.

En ambos casos se aplica la restricción TR ⊂ HS, y las filas de test con `#¿NOMBRE?` (sin texto recuperable) reciben la clase mayoritaria `0,0,0`.

In [26]:
B_tr_f, B_te_f, _ = ajustar_bloques(TXT_TR, TXT_TE, L_TR, L_TE)

def predecir_test(nombre):
    bloques = BLOQUES_BASE.get(nombre, TODOS)
    return predecir(nombre, unir(B_tr_f, bloques), Y, COMBO, unir(B_te_f, bloques))

if FINAL == 'stacking':
    Z_test = np.hstack([predecir_test(k) for k in BASES_STACK])
    P_test = np.column_stack([meta_modelo().fit(Z_OOF, Y[:, j]).predict_proba(Z_test)[:, 1]
                              for j in range(len(LABELS))])
    descripcion = f'stacking sobre {BASES_STACK}'
else:
    P_test = np.mean([predecir_test(k) for k in ENSAMBLE], axis=0)
    descripcion = f'ensamble {ENSAMBLE}'
Y_test = binarizar(P_test, UMBRALES_FINAL)

err_excel = test['text'].str.strip().str.match(RE_ERROR_EXCEL).values
Y_test[err_excel] = 0
log(f'Modelo final: {descripcion} | umbrales {dict(zip(LABELS, UMBRALES_FINAL))} | '
    f'filas #¿NOMBRE? en test: {err_excel.sum()}')

pred = pd.DataFrame(Y_test, columns=LABELS)
pred.insert(0, 'id', test['id'].values)
submission = sample[['id']].merge(pred, on='id', how='left')

# Verificaciones de formato
assert list(submission.columns) == list(sample.columns) == ['id', 'HS', 'TR', 'AG']
assert len(submission) == len(sample) == len(test)
assert set(submission['id']) == set(test['id'])
assert submission[LABELS].notna().all().all() and submission[LABELS].isin([0, 1]).all().all()
assert not ((submission.TR == 1) & (submission.HS == 0)).any()
submission[LABELS] = submission[LABELS].astype(int)
submission.to_csv(OUT_PATH, index=False)
log('Submission guardada en', os.path.abspath(OUT_PATH))

display(pd.DataFrame({'prevalencia train': train[LABELS].mean(),
                      'prevalencia predicha en test': submission[LABELS].mean()}).round(3))
submission.head()

[    61s] Modelo final: stacking sobre ['CNB', 'LR', 'SVM', 'NB-LR', 'Powerset'] | umbrales {'HS': np.float64(0.5), 'TR': np.float64(0.5), 'AG': np.float64(0.5)} | filas #¿NOMBRE? en test: 6


[    61s] Submission guardada en C:\Users\jesus.romerom\Documents\Maestria\PLN\natural-language-processing-workshops\workshops\Kaggle-tweet-toxic\toxic_submission_v2.csv


,prevalencia train,prevalencia predicha en test
HS,0.437,0.393
TR,0.300,0.298
AG,0.382,0.373


,id,HS,TR,AG
0,20929,0,0,0
1,22807,1,1,1
2,24595,0,0,0
3,23092,0,0,0
4,22129,0,0,0


## 11. Conclusiones
1. **Leer los datos antes de modelar fue lo que más aportó.** El bloque de léxico construido con las hipótesis sube el F1 medio de 0,781 a 0,803 sobre la misma regresión logística, sobre todo en AG (+4,2 puntos) y HS (+1,6). Los ajustes de hiperparámetros, en cambio, mueven el resultado menos de medio punto (sección 8.8).
2. **Hipótesis.** H1-H10 se confirman con los datos (sección 6.3). Las más discriminativas son las que resuelven la ambigüedad de las palabras clave con las que se armó el corpus: *puta* como intensificador (H2), el **género del blanco** (H3: *hijo* frente a *hija de puta*, concordancia *el*/*la*) y *cállate* sin insulto femenino (H4). H11 se confirma a medias: los n-gramas de caracteres solos rinden menos que las palabras (0,757 frente a 0,771), pero combinados suman un punto.
3. **La estructura entre etiquetas importa.** El modelo *label powerset* es el mejor modelo individual (F1 0,806, EMR 0,780) porque aprende combinaciones como "chiste misógino = `100`" o "insulto directo a una mujer = `111`". El stacking, que aprende a combinar los cinco modelos y a usar unas etiquetas como evidencia de otras, es el mejor en total (0,815), y la restricción TR ⊂ HS garantiza predicciones coherentes.
4. **Elegir bien la métrica importa tanto como el modelo.** La v1 se optimizó y se reportó con el F1-macro de dos clases, que promedia también la clase 0 y por eso da 4 puntos más. Eso hizo parecer bajo un puntaje de Kaggle (0,785) que en realidad estaba dentro de lo esperado. La v2 se elige con el F1 de la clase positiva, y el stacking cambia 59 predicciones del test, casi todas de 0 a 1 (más cobertura de la clase positiva, que es lo que esta métrica premia).
5. **Decisiones conservadoras.** Ajustar umbrales por etiqueta no mejoró la estimación honesta (CV anidada), así que se mantuvo 0,5. El ensamble se eligió entre pocas combinaciones predefinidas, y el stacking solo se adopta si supera al ensamble por un margen mínimo. Con ~577 filas en el leaderboard público, el ruido de muestreo es de unos ±2,5 puntos, mayor que cualquier mejora entre versiones; por eso las decisiones se toman con la CV y no con el leaderboard.

**Limitaciones**
- **Ruido de anotación**: *perra*/*zorra* en tono afectuoso se etiqueta de forma inconsistente, y eso pone un techo a cualquier modelo de bolsa de palabras.
- **Artefacto de comillas (H10)**: el modelo lo aprende como señal de AG. Si en el test (25 filas entrecomilladas) el artefacto no se mantiene, AG se vería afectado.
- **Pérdida de tildes y muestreo por palabras clave**: el modelo aprende sobre todo contextos de esas palabras. El odio sin palabras clave (estereotipos implícitos, ironía) es el error más frecuente.
- La prevalencia de HS predicha en test (≈ 0,39) es menor que la del train (0,44). El test tiene menos vocabulario xenófobo explícito (insultos de grupo migrante: 3,0 % frente a 5,4 % de los tweets), así que parte de la diferencia es real y no una subpredicción.

**Trabajo futuro**
- Ampliar las marcas de concordancia de género con un etiquetador morfosintáctico entrenado desde cero.
- Añadir léxicos de insultos públicos (p. ej. HurtLex) como variables, siempre que el curso los acepte.
- Confirmar en la pestaña *Evaluation* de Kaggle la definición exacta del F1 (clase positiva, micro o *samples*) y ajustar la métrica de selección si es distinta.